# RL dry-run rollout

Sample GSM8K completions from the LLM project's Qwen2.5-3B base SFT adapter and inspect reward variance before RL training.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os

candidates = [
    Path('/content/drive/MyDrive/LLM_project'),
    Path('/content/drive/MyDrive/LLM_project/project'),
    Path('/content/project'),
    Path.cwd(),
]
PROJECT_DIR = next((p for p in candidates if (p / 'RL_dryrun_rollout').exists() and (p / 'RL_common').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Cannot find LLM project root. Mount Drive or edit PROJECT_DIR manually.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
!pwd

PROJECT_DIR = /content/drive/MyDrive/LLM_project
/content/drive/MyDrive/LLM_project


## Optional token

In [3]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

print('HF_TOKEN set =', bool(os.environ.get('HF_TOKEN')))

HF_TOKEN set = True


## Install local package

In [ ]:
!pip install -q -e RL_common

## Run rollout

In [6]:
!pip uninstall -y torchao
ACTIVE_CONFIG = PROJECT_DIR / 'RL_dryrun_rollout/configs/qwen25_3b_sft_dryrun.yaml'
print(ACTIVE_CONFIG.read_text(encoding='utf-8'))
!python RL_dryrun_rollout/run_rollout.py --config "{ACTIVE_CONFIG}"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
run:
  name: qwen25_3b_base_sft_dryrun_g{generation.num_generations}_train{dataset.limit}
  output_root: RL_dryrun_rollout/outputs

model:
  base_model_name_or_path: Qwen/Qwen2.5-3B
  adapter_path: SFT_train/outputs/qwen25_3b_base_gsm8k_lora_sft_full
  tokenizer_name_or_path: SFT_train/outputs/qwen25_3b_base_gsm8k_lora_sft_full
  trust_remote_code: true
  prompt_template: qwen_chat
  system_prompt: ""
  include_empty_system: false
  eos_tokens:
    - <|im_end|>
    - <|endoftext|>
  dtype: bf16
  device_map: auto
  load_in_4bit: false
  is_trainable_adapter: false

dataset:
  kind: gsm8k_train_subset
  name: openai/gsm8k
  config: main
  start_index: 0
  limit: 200

generation:
  num_generations: 16
  batch_size: 512
  max_prompt_length: 512
  max_new_tokens: 256
  do_sample: true
  temperature: 1.0
  top_p: 0.95

reward:
  answer_correct: 1.0
  answer_incorrect: 0.0
  for

## Inspect summary

In [7]:
import json
import sys

COMMON_SRC = PROJECT_DIR / 'RL_common/src'
if str(COMMON_SRC) not in sys.path:
    sys.path.insert(0, str(COMMON_SRC))

from rl_common.config import format_run_name, load_config, make_run_dir

cfg = load_config(str(ACTIVE_CONFIG))
cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
print('run_dir =', run_dir)
summary_path = run_dir / 'summary.json'
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))

run_dir = /content/drive/MyDrive/LLM_project/RL_dryrun_rollout/outputs/qwen25_3b_base_sft_dryrun_g16_train200
{
  "prompt_count": 200,
  "completion_count": 3200,
  "all_correct_rate": 0.3,
  "all_wrong_rate": 0.025,
  "mixed_rate": 0.675,
  "parse_fail_rate": 0.000313,
  "eos_stop_rate": 0.817813,
  "avg_completion_length": 145.882812,
  "reward_distribution": {
    "mean": 0.822375,
    "std": 0.439642,
    "min": -0.15,
    "max": 1.2
  }
}
